# 12_phase4_GSE114725_sensitivity.ipynb
Phase 4 — QC threshold and normalisation sensitivity analysis (GSE114725)

Same design as the GSE176078 sensitivity notebooks (10, 11), applied to GSE114725. Baseline QC threshold for this dataset is n_genes>200 (vs GSE176078's >500) — loose/strict alternatives scaled proportionally: >100 (loose) and >350 (strict).

**Headline finding checked:** macrophage reprogramming (tumour vs normal) — specifically whether FN1/HSPA1A/B remain elevated in tumour macrophages relative to normal, using a lightweight marker-based check rather than full re-running pseudobulk DE at each configuration (keeps this notebook's runtime reasonable).

**Memory management:** same lesson learned from notebook 10 — process one configuration fully, extract what's needed, free the object, before starting the next.

In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import os
os.environ["NUMBA_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import scrublet as scr
import matplotlib.pyplot as plt
from scipy.sparse import issparse, csr_matrix, vstack
from pathlib import Path

sc.settings.verbosity = 1

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RAW_DIR = PROJECT_DIR / "Data" / "Raw"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_sensitivity"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase4_sensitivity"

for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [2]:
# ----------------------------
# Cell 2 — Reusable pipeline function for GSE114725, parameterised by
# QC threshold AND normalisation target_sum (tests both variables via
# this one function, called with different arguments)
# ----------------------------
def run_pipeline_ge114725(n_genes_threshold, target_sum, label):
    print(f"\n{'='*70}")
    print(f"GSE114725 — n_genes>{n_genes_threshold}, target_sum={target_sum} ({label})")
    print(f"{'='*70}\n")

    adata = sc.read_h5ad(RAW_DIR / "GSE114725_raw.h5ad")
    if issparse(adata.X):
        adata.X = adata.X.tocsr().astype("float32", copy=False)
    else:
        adata.X = csr_matrix(adata.X, dtype=np.float32)
    gc.collect()

    vdj_prefixes = ("IGHV", "IGLV", "IGKV", "TRAV", "TRBV", "TRGV", "TRDV",
                     "IGHD", "IGHJ", "IGLJ", "IGKJ")
    constant_genes = ("TRAC", "TRBC", "TRGC", "TRDC", "IGHA", "IGHD", "IGHE",
                       "IGHG", "IGHM", "IGLC", "IGKC")
    all_prefixes = vdj_prefixes + constant_genes
    vdj_mask = ~adata.var_names.str.startswith(all_prefixes)
    mt_mask = ~adata.var_names.str.startswith("MT-")
    adata = adata[:, vdj_mask & mt_mask].copy()
    gc.collect()

    n_cells = adata.n_obs
    n_genes_by_counts = np.zeros(n_cells, dtype=np.float32)
    chunk_size = 2000
    for start in range(0, n_cells, chunk_size):
        end = min(start + chunk_size, n_cells)
        chunk = adata.X[start:end]
        n_genes_by_counts[start:end] = np.asarray((chunk > 0).sum(axis=1)).flatten()
    adata.obs["n_genes_by_counts"] = n_genes_by_counts

    n_before = adata.n_obs
    adata = adata[adata.obs["n_genes_by_counts"] > n_genes_threshold].copy()
    sc.pp.filter_genes(adata, min_cells=3)
    n_after = adata.n_obs
    print(f"Cell filtering: {n_before} -> {n_after} cells")
    gc.collect()

    all_predicted_doublets = np.zeros(adata.n_obs, dtype=bool)
    samples = adata.obs["patient"].unique()
    for sample in samples:
        sample_mask = adata.obs["patient"] == sample
        sample_idx = np.where(sample_mask)[0]
        if len(sample_idx) < 50:
            continue
        chunks = []
        for start in range(0, len(sample_idx), chunk_size):
            end = min(start + chunk_size, len(sample_idx))
            global_idx = sample_idx[start:end]
            chunk = adata.X[global_idx]
            if not issparse(chunk): chunk = csr_matrix(chunk)
            chunks.append(chunk)
        X_sample = vstack(chunks)
        try:
            scrub = scr.Scrublet(X_sample)
            _, predicted_doublets = scrub.scrub_doublets(verbose=False)
            all_predicted_doublets[sample_idx] = predicted_doublets
        except Exception as e:
            print(f"  Scrublet failed for {sample}: {e}")
        gc.collect()

    adata.obs["predicted_doublet"] = all_predicted_doublets
    before_doublets = adata.n_obs
    adata = adata[~adata.obs["predicted_doublet"]].copy()
    print(f"Doublets removed: {before_doublets - adata.n_obs} (final: {adata.n_obs} cells)")
    gc.collect()

    sc.pp.normalize_total(adata, target_sum=target_sum)
    sc.pp.log1p(adata)
    adata.raw = adata

    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
    adata = adata[:, adata.var.highly_variable].copy()
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack", random_state=0)
    sce.pp.harmony_integrate(adata, key="patient", basis="X_pca", random_state=0)
    gc.collect()

    sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30, random_state=0)
    sc.tl.leiden(adata, resolution=0.2, key_added="leiden_0.2",
                 flavor="igraph", n_iterations=2, directed=False, random_state=0)
    n_clusters = adata.obs["leiden_0.2"].nunique()
    print(f"\nFinal: {adata.n_obs} cells, {n_clusters} clusters at resolution 0.2")

    return adata, n_clusters

def check_macrophage_reprogramming(adata, label):
    """Lightweight check: are FN1/HSPA1A/B elevated in tumour vs normal
    macrophages, matching the validated pseudobulk DE finding?"""
    adata_raw = adata.raw.to_adata()
    genes_present = [g for g in ["FN1", "HSPA1A", "HSPA1B", "CD68", "LYZ"] if g in adata_raw.var_names]
    if not genes_present or "tissue" not in adata.obs.columns:
        print(f"  {label}: required genes/metadata not found, skipping")
        return None

    # crude macrophage identification: CD68+ cells, avoids needing full re-annotation
    if "CD68" not in adata_raw.var_names:
        print(f"  {label}: CD68 not found, cannot identify macrophages, skipping")
        return None
    cd68 = adata_raw[:, "CD68"].X.toarray().flatten()
    is_macrophage_like = cd68 > 1.0
    adata.obs["macrophage_like"] = is_macrophage_like
    mac_mask = adata.obs["macrophage_like"].values

    results = {}
    for gene in [g for g in ["FN1", "HSPA1A", "HSPA1B"] if g in adata_raw.var_names]:
        expr = adata_raw[:, gene].X.toarray().flatten()
        tumor_mask = mac_mask & (adata.obs["tissue"] == "TUMOR").values
        normal_mask = mac_mask & (adata.obs["tissue"] == "NORMAL").values
        if tumor_mask.sum() < 10 or normal_mask.sum() < 10:
            print(f"  {label}: too few macrophage-like cells for {gene} check")
            continue
        tumor_mean = expr[tumor_mask].mean()
        normal_mean = expr[normal_mask].mean()
        results[gene] = {"tumor_mean": tumor_mean, "normal_mean": normal_mean,
                          "elevated_in_tumor": tumor_mean > normal_mean}
        print(f"  {label}: {gene} - tumor={tumor_mean:.3f}, normal={normal_mean:.3f}, "
              f"elevated in tumor: {tumor_mean > normal_mean}")
    return results

print("Pipeline functions ready")

Pipeline functions ready


In [3]:
# ----------------------------
# Cell 3 — LOOSE threshold (n_genes>100, baseline normalisation target_sum=1e4)
# Baseline for comparison: n_genes>200, 44,662 cells, 9 clusters
# ----------------------------
adata_loose, n_clusters_loose = run_pipeline_ge114725(100, 1e4, "LOOSE QC threshold")
result_loose = check_macrophage_reprogramming(adata_loose, "Loose QC (>100)")
if result_loose:
    pd.DataFrame(result_loose).T.to_csv(RESULTS_DIR / "GSE114725_loose_QC_macrophage_check.csv")
n_cells_loose = adata_loose.n_obs
del adata_loose
gc.collect()
print("\nLoose-threshold object freed. Ready for next configuration.")


GSE114725 — n_genes>100, target_sum=10000.0 (LOOSE QC threshold)

Cell filtering: 47016 -> 46943 cells
Doublets removed: 870 (final: 46073 cells)


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-07-14 16:25:39,712 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-14 16:26:06,364 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-14 16:26:06,870 - harmonypy - INFO - Iteration 1 of 10
2026-07-14 16:26:44,473 - harmonypy - INFO - Iteration 2 of 10
2026-07-14 16:27:15,282 - harmonypy - INFO - Converged after 2 iterations
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Final: 46073 cells, 9 clusters at resolution 0.2
  Loose QC (>100): FN1 - tumor=1.769, normal=0.046, elevated in tumor: True
  Loose QC (>100): HSPA1A - tumor=1.091, normal=0.251, elevated in tumor: True
  Loose QC (>100): HSPA1B - tumor=0.403, normal=0.040, elevated in tumor: True

Loose-threshold object freed. Ready for next configuration.


In [4]:
# ----------------------------
# Cell 4 — STRICT threshold (n_genes>350, baseline normalisation)
# ----------------------------
adata_strict, n_clusters_strict = run_pipeline_ge114725(350, 1e4, "STRICT QC threshold")
result_strict = check_macrophage_reprogramming(adata_strict, "Strict QC (>350)")
if result_strict:
    pd.DataFrame(result_strict).T.to_csv(RESULTS_DIR / "GSE114725_strict_QC_macrophage_check.csv")
n_cells_strict = adata_strict.n_obs
del adata_strict
gc.collect()
print("\nStrict-threshold object freed. Ready for next configuration.")


GSE114725 — n_genes>350, target_sum=10000.0 (STRICT QC threshold)

Cell filtering: 47016 -> 36600 cells
Doublets removed: 634 (final: 35966 cells)


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-07-14 16:32:14,637 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-14 16:32:32,242 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-14 16:32:32,625 - harmonypy - INFO - Iteration 1 of 10
2026-07-14 16:32:55,881 - harmonypy - INFO - Iteration 2 of 10
2026-07-14 16:33:18,432 - harmonypy - INFO - Converged after 2 iterations



Final: 35966 cells, 10 clusters at resolution 0.2
  Strict QC (>350): FN1 - tumor=1.810, normal=0.051, elevated in tumor: True
  Strict QC (>350): HSPA1A - tumor=1.117, normal=0.260, elevated in tumor: True
  Strict QC (>350): HSPA1B - tumor=0.415, normal=0.044, elevated in tumor: True

Strict-threshold object freed. Ready for next configuration.


In [5]:
# ----------------------------
# Cell 5 — Alternative normalisation (target_sum=None, baseline QC n_genes>200)
# ----------------------------
adata_median_norm, n_clusters_median = run_pipeline_ge114725(200, None, "MEDIAN normalisation")
result_median = check_macrophage_reprogramming(adata_median_norm, "Median normalisation")
if result_median:
    pd.DataFrame(result_median).T.to_csv(RESULTS_DIR / "GSE114725_median_norm_macrophage_check.csv")
n_cells_median = adata_median_norm.n_obs
del adata_median_norm
gc.collect()
print("\nMedian-normalisation object freed.")


GSE114725 — n_genes>200, target_sum=None (MEDIAN normalisation)

Cell filtering: 47016 -> 45311 cells
Doublets removed: 649 (final: 44662 cells)


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-07-14 16:36:44,606 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-14 16:37:08,072 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-14 16:37:08,536 - harmonypy - INFO - Iteration 1 of 10
2026-07-14 16:37:37,042 - harmonypy - INFO - Iteration 2 of 10
2026-07-14 16:38:03,961 - harmonypy - INFO - Converged after 2 iterations



Final: 44662 cells, 10 clusters at resolution 0.2
  Median normalisation: FN1 - tumor=1.189, normal=0.010, elevated in tumor: True
  Median normalisation: HSPA1A - tumor=0.349, normal=0.083, elevated in tumor: True
  Median normalisation: HSPA1B - tumor=0.088, normal=0.015, elevated in tumor: True

Median-normalisation object freed.


In [6]:
# ----------------------------
# Cell 6 — Full comparison summary
# Baseline (n_genes>200, target_sum=1e4, validated): 44,662 cells, 9 clusters
# ----------------------------
comparison = pd.DataFrame({
    "Configuration": ["Loose QC (>100)", "Baseline (>200, 1e4)", "Strict QC (>350)", "Median norm (target_sum=None)"],
    "Cells": [n_cells_loose, 44662, n_cells_strict, n_cells_median],
    "Clusters (res 0.2)": [n_clusters_loose, 9, n_clusters_strict, n_clusters_median],
})
print(comparison.to_string(index=False))
comparison.to_csv(RESULTS_DIR / "GSE114725_sensitivity_summary.csv", index=False)

print("\n>>> Check the three saved CSVs (loose/strict/median macrophage checks) for")
print(">>> whether FN1/HSPA1A/B remain elevated in tumour vs normal macrophages")
print(">>> across all configurations — this is the robustness test for the")
print(">>> macrophage reprogramming finding, GSE114725's equivalent to the")
print(">>> HER2+ Memory T cell check done for GSE176078.")

                Configuration  Cells  Clusters (res 0.2)
              Loose QC (>100)  46073                   9
         Baseline (>200, 1e4)  44662                   9
             Strict QC (>350)  35966                  10
Median norm (target_sum=None)  44662                  10

>>> Check the three saved CSVs (loose/strict/median macrophage checks) for
>>> whether FN1/HSPA1A/B remain elevated in tumour vs normal macrophages
>>> across all configurations — this is the robustness test for the
>>> macrophage reprogramming finding, GSE114725's equivalent to the
>>> HER2+ Memory T cell check done for GSE176078.
